<img src="../../../At-Home-Round/Weather/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), manche a domicile](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/At-Home-Round/Weather/Weather.ipynb)

# Prevision meteo a partir d'images satellite

Veuillez d'abord utiliser Google Colab pour verifier votre solution.
Nous nous excusons pour la difficulte de soumission : la Chine continentale ne pouvant pas acceder directement a huggingface.co, il existe trop d'obstacles reseau entre le serveur de test, huggingface.co et hf-mirror.com. Nous identifions actuellement le probleme, ce qui peut prendre du temps.

Le lien Google Colab de cette tache est https://colab.research.google.com/drive/14tI_EARXubr6NNl7T_Ikhz0A5-Jhy2hZ?usp=sharing. Les concurrents peuvent telecharger les jeux de donnees et deboguer hors ligne. Il est neanmoins fortement recommande d'utiliser Bohrium pour se familiariser avec la plateforme du concours, car la manche sur site ne fournira pas de liens Colab.

Vous etes un lyceen en stage dans un laboratoire climatique regional. Vous travaillez avec une petite equipe de scientifiques qui cherchent a ameliorer la prevision des pluies a partir d'images satellites uniquement. Traditionnellement, la pluie est mesuree par radar et capteurs au sol, mais ces systemes sont couteux et souvent indisponibles dans les regions reculees.

<img src="../../../At-Home-Round/Weather/figs/Weather Fig 1.png" width="400">

En analysant les images du satellite GOES-16, une idee vous vient :

> "Et si l'on entrainait un modele pour detecter la pluie directement a partir des images satellites, sans donnees au sol ? Et si on ajoutait des informations contextuelles comme l'angle solaire, l'heure et la localisation pour ameliorer la precision ?"

Les scientifiques sont interesses. Vous avez acces a une vaste archive de donnees satellites et de masques de precipitation. Un defi : prouver que cela peut marcher. En cas de succes, votre approche aiderait les petits agriculteurs des regions isolees a mieux planifier l'irrigation et a reduire les pertes.

Votre mission : construire un modele qui regarde le ciel et dit s'il va pleuvoir.


## Tache

Vous devez developper un modele d'IA qui prend en entree l'imagerie satellite du GOES-16 et predit, pour chaque pixel, si la pluie est en train de tomber a l'endroit correspondant sur Terre. C'est une tache de **segmentation semantique** : votre modele doit produire un masque binaire indiquant `pluie` ou `non-pluie` pour chaque pixel.

Vous pouvez entrainer et evaluer votre modele en utilisant les donnees de precipitations de reference fournies par le jeu MRMS (Multi-Radar Multi-Sensor).

Un modele de segmentation baseline base sur un U-Net pre-entraine est fourni pour vous aider a demarrer.

**Vous AVEZ le droit de :**
- fine-tuner ou modifier le U-Net baseline ;
- utiliser les metadonnees (latitude, longitude, temps, elevation solaire) ;
- modifier la logique d'inference (seuils, post-traitement).

**Vous N'AVEZ PAS le droit de :**
- utiliser un jeu de donnees externe ;
- utiliser des modeles pre-entraines externes autres que le baseline fourni ;
- chercher des articles de prevision meteo sur Internet. Bien que ce soit une tache a domicile, elle prepare au concours sur site.

## Donnees

Le jeu de donnees comprend des observations satellites de 2024 :

- **Imagerie multicanal GOES-16 ABI**
  16 canaux spectraux (C01-C16) couvrant plusieurs longueurs d'onde :
  - C01-C03 : lumiere visible ;
  - C04-C06 : proche infrarouge ;
  - C07-C16 : infrarouge.
  Les images sont decoupees en patches de 128x128 ou 256x256 pixels.

- **Masques de precipitation** du systeme MRMS, fournissant des labels binaires (pluie/pas pluie) par pixel.

- **Metadonnees** :
  - latitude et longitude du coin superieur gauche du patch ;
  - heure de debut et de fin en UTC (la capture des 16 canaux prend environ 10 minutes) ;
  - une fonction Python pour calculer l'**elevation solaire** en fonction de l'heure et du lieu.


### Split entrainement/validation

- L'**ensemble d'entrainement** est biaise vers les scenes pluvieuses : seuls les patches dont au moins 3% des pixels contiennent de la pluie sont inclus.
  
- L'**ensemble de validation** reflete les conditions reelles : beaucoup de patches contiennent peu ou pas de pluie. Certains echantillons peuvent contenir des problemes de transmission (canaux manquants ou corrompus, par exemple).


## Evaluation

Votre modele est evalue sur deux metriques :

- **Dice moyen** :
  Mesure a quel point votre masque predit recouvre la verite terrain, pixel par pixel. Formule : 2 x |intersection| / (|prediction| + |verite|).

- **Precision au niveau image (pluie/pas pluie)** :
  Mesure si votre modele classe correctement la presence de pluie dans l'image. Parfois il n'est pas necessaire de segmenter chaque goutte : il suffit de savoir s'il pleut. Une seule prediction precise peut suffire a proteger un champ entier.

- **Score final** :
  Score final = (Dice moyen + Precision au niveau image) / 2.

## Droits

Toutes les donnees utilisees sont publiques :

- **GOES-16 ABI**, donnees de la NOAA et du NESDIS ;
- **MRMS**, donnees de precipitation du National Severe Storms Laboratory (NSSL) de la NOAA.


## Soumission

Votre notebook doit generer un fichier `submission.zip` contenant vos predictions pour le jeu de test public `pred_a.npz` et pour le jeu de test prive `pred_b.npz`. Chaque `.npz` doit contenir `Y_pred_128` (forme $51 \times 128 \times 128$) et `Y_pred_256` (forme $183 \times 256 \times 256$), vos predictions booleennes pour chaque jeu de test.

```python
# genere d'abord pred_a.npz

model.eval()
model.to(DEVICE)

Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i]
        metadata = df[(df['size'] == 128) & (df['split'] == 'test') & (df['ind'] == i)] # exemple d'utilisation des metadonnees
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i]
        metadata = df[(df['size'] == 256) & (df['split'] == 'test') & (df['ind'] == i)]
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# Vos tableaux de prediction DOIVENT s'appeler `Y_pred_128` et `Y_pred_256`, et le fichier `pred_a.npz` pour le classement public
np.savez('pred_a.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)
```

```
100%|##########| 51/51 [00:00<00:00, 53.08it/s]
(51, 128, 128)
100%|##########| 183/183 [00:08<00:00, 21.31it/s]
(183, 256, 256)
```

```python
# Le notebook accedera au jeu de test via la variable d'environnement DATA_PATH apres soumission.
TEST_PATH = os.environ.get('DATA_PATH', "test")

test = np.load(Path(TEST_PATH) / "X_test.npz") # Lit les donnees de test depuis X_test.npz dans le chemin fourni

X_test = {
    128: torch.from_numpy(test['X_test_128']),
    256: torch.from_numpy(test['X_test_256']),
} # seul X_test est fourni

df_test = pd.read_csv(Path(TEST_PATH) / "metadata_test.csv") # Lit les metadonnees depuis metadata_test.csv
```

```python
Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i]
        metadata = df_test[(df_test['size'] == 128) & (df_test['ind'] == i)]
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i]
        metadata = df_test[(df_test['size'] == 256) & (df_test['ind'] == i)]
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# Nommez `pred_b.npz` pour le classement prive
np.savez('pred_b.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)
```
```
100%|##########| 51/51 [00:00<00:00, 58.14it/s]
(51, 128, 128)
100%|##########| 183/183 [00:08<00:00, 22.31it/s]
(183, 256, 256)
```

```python
# compresse `pred_a.npz` et `pred_b.npz` dans `submission.zip`
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('pred_a.npz')
    zipf.write('pred_b.npz')
```



## Imports


In [ ]:
import numpy as np
import torch
import math
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path
import zipfile
from tqdm import tqdm
from datetime import datetime, timedelta
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

TEST_PATH = "/bohr/train-ma50/v2/"
DATASET_PATH =  TEST_PATH + "dataset.npz"
METADATA_PATH = TEST_PATH + "metadata.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASELINE_MODEL = TEST_PATH + "model_weights.pth"

## Fonctions utilitaires sur les donnees


In [ ]:
def pixel_to_latlon(i, j):
    """
    Converts pixel indices (row i, column j) to geographic latitude and longitude.

    Args:
        i (int): Row index (from 0 at the top to 3499 at the bottom).
        j (int): Column index (from 0 at the left to 6999 at the right).

    Returns:
        tuple: (latitude, longitude) as floats.

    This function is used to convert from image/pixel coordinates 
    (such as a satellite image) to actual map coordinates.
    """
    lat = 60.0 - i * 0.01   # Each row down is 0.01 degree further south
    lon = -130.0 + j * 0.01 # Each column right is 0.01 degree further east
    return lat, lon


def solar_elevation(x, y, dt_utc):
    """
    Calculates the sun's elevation angle above the horizon for a specific
    location (pixel) and UTC time.

    Args:
        x (int): Row index in the image (vertical position).
        y (int): Column index in the image (horizontal position).
        dt_utc (datetime or str): The date and time in UTC (string or datetime).

    Returns:
        float: Solar elevation angle in degrees.

    This function tells you "how high is the sun in the sky" 
    for a given place and time.
    """
    # If time is given as string, convert to datetime
    if isinstance(dt_utc, str):
        dt_utc = datetime.strptime(dt_utc, '%Y-%m-%d %H:%M:%S.%f')

    # Convert pixel indices to latitude and longitude
    lat, lon = pixel_to_latlon(x, y)

    # Estimate local time (in hours) by longitude (15 degrees = 1 hour)
    timezone_offset = lon / 15.0
    local_time = dt_utc.hour + dt_utc.minute / 60 + timezone_offset

    # Day of year (1-365/366)
    N = dt_utc.timetuple().tm_yday

    # Solar declination: angle between sun's rays and Earth's equator
    decl = 23.44 * math.sin(math.radians(360 / 365 * (N - 81)))

    # Hour angle: how far in time from solar noon
    H = 15 * (local_time - 12)  # degrees

    # Convert everything to radians for math functions
    phi = math.radians(lat)
    delta = math.radians(decl)
    H = math.radians(H)

    # Calculate elevation using spherical trigonometry
    sin_h = math.sin(phi) * math.sin(delta) + math.cos(phi) * math.cos(delta) * math.cos(H)
    h = math.degrees(math.asin(sin_h))
    return h


def parse_goes_time(timestr):
    """
    Converts a GOES satellite timestamp string into a Python datetime.

    Args:
        timestr (str): Timestamp string, e.g. 's20242891100205'.

    Returns:
        datetime: The parsed datetime.

    Format explanation:
        - 's20242891100205' means:
            year = 2024,
            day-of-year = 289,
            hour = 11,
            minute = 00,
            second = 20,
            tenths of a second = 5
    """
    year = int(timestr[1:5])
    doy = int(timestr[5:8])
    hour = int(timestr[8:10])
    minute = int(timestr[10:12])
    second = int(timestr[12:14])
    micro = int(timestr[14]) * 100000  # tenths of a second
    return datetime(year, 1, 1) + timedelta(days=doy - 1, hours=hour, minutes=minute, seconds=second, microseconds=micro)


def prepare_dataset(data, threshold=0.1):
    """
    Prepares input (X) and output (Y) tensors from raw patch data,
    applying normalization, thresholding, and cleaning.

    Args:
        data (dict): Dictionary mapping patch size to lists of numpy arrays,
                     each array is [17, D, D] (16 channels + 1 mask channel).
        threshold (float): Threshold for converting last channel to binary mask.

    Returns:
        tuple: (X_dict, Y_dict, norm_stats)
            - X_dict: Dict of input tensors, shape [N, 16, D, D] for each size.
            - Y_dict: Dict of output masks, shape [N, D, D] for each size.
            - norm_stats: Dict with 'mean' and 'std' for normalization (per channel).

    This function does two passes over the data:
        1. Computes mean and std for each input channel (ignoring NaNs).
        2. Normalizes the data and packs it into torch tensors.
    """

    num_channels = 16  # First 16 channels are features, last one is target mask

    # === Pass 1: Compute statistics for normalization ===
    sum_channels = torch.zeros(num_channels, dtype=torch.float64)    # Total sum per channel
    sum_sq_channels = torch.zeros(num_channels, dtype=torch.float64) # Total squared sum per channel
    count_channels = torch.zeros(num_channels, dtype=torch.int64)    # Number of valid (non-NaN) values per channel

    for patches in data.values():
        for arr in patches:
            arr = torch.tensor(arr.squeeze(0), dtype=torch.float32)  # arr: [17, D, D]
            x = arr[:16]  # [16, D, D] - input features
            x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)  # Replace NaN/inf with 0

            valid = ~torch.isnan(x)  # Mask of valid values
            sum_channels += torch.where(valid, x, torch.tensor(0.0)).sum(dim=(1, 2))
            sum_sq_channels += torch.where(valid, x ** 2, torch.tensor(0.0)).sum(dim=(1, 2))
            count_channels += valid.sum(dim=(1, 2))

    # Compute mean and std per channel
    means = sum_channels / count_channels
    variances = (sum_sq_channels / count_channels) - means ** 2
    stds = torch.sqrt(torch.clamp(variances, min=1e-6))  # avoid sqrt of negative

    # === Make sure no zeros, NaNs or infs in std/mean ===
    stds[stds == 0] = 1.0
    stds[torch.isnan(stds)] = 1.0
    stds[torch.isinf(stds)] = 1.0

    means[torch.isnan(means)] = 0.0
    means[torch.isinf(means)] = 0.0

    norm_stats = {
        "mean": means.to(torch.float32),
        "std": stds.to(torch.float32)
    }

    # === Pass 2: Normalize and pack tensors for PyTorch training ===
    X_dict = {}
    Y_dict = {}

    for size, patches in data.items():
        n = len(patches)    # Number of patches for this size
        D = size            # Patch size (width and height)

        # Allocate memory for all normalized patches and masks
        X_tensor = torch.empty((n, num_channels, D, D), dtype=torch.float16)
        Y_tensor = torch.empty((n, D, D), dtype=torch.uint8)

        for i, arr in enumerate(patches):
            arr = torch.tensor(arr.squeeze(0), dtype=torch.float32)  # [17, D, D]
            x = arr[:16]           # First 16 channels: features
            y = (arr[16] > threshold).to(torch.uint8)  # Last channel: mask, binarized

            x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)  # Clean

            # Normalize per channel: (value - mean) / std
            x_norm = ((x - means[:, None, None]) / stds[:, None, None]).to(torch.float16)

            X_tensor[i] = x_norm   # Save normalized input
            Y_tensor[i] = y        # Save output mask

        X_dict[size] = X_tensor
        Y_dict[size] = Y_tensor

    return X_dict, Y_dict, norm_stats

## Fonctions utilitaires sur les modeles


In [ ]:
class PatchDataset(Dataset):
    """
    Custom PyTorch dataset for image patches and their segmentation masks.

    Args:
        X_tensor (torch.Tensor): Input tensor of image patches, shape [N, 16, D, D]
        Y_tensor (torch.Tensor): Target segmentation masks, shape [N, D, D]

    This class is used to feed data into the neural network during training and testing.
    """
    def __init__(self, X_tensor, Y_tensor):
        self.X = X_tensor  # Input images: shape [num_samples, 16, height, width], float16
        self.Y = Y_tensor  # Target masks: shape [num_samples, height, width], uint8 (0 or 1)

    def __len__(self):
        # Returns number of samples in dataset
        return self.X.shape[0]

    def __getitem__(self, idx):
        """
        Fetch one sample and its mask by index.
        Converts X to float32 and adds a channel to Y (needed for loss).
        """
        x = self.X[idx].to(torch.float32)  # Convert to float32 (needed for the model)
        y = self.Y[idx].unsqueeze(0).to(torch.float32)  # Add a channel, shape [1, D, D]
        return x, y


class UNetSegmentation(nn.Module):
    """
    U-Net neural network for image segmentation.

    Args:
        in_channels (int): Number of input channels (e.g., 16 for your patches)

    The U-Net consists of an encoder (downsampling), a bottleneck (middle), 
    and a decoder (upsampling). It is widely used for segmenting images, 
    where every pixel needs to be classified (e.g., rain/no rain).
    """
    def __init__(self, in_channels):
        super().__init__()
        # Encoder path ("contracting" path): extracts features, reduces size
        self.encoder1 = self.conv_block(in_channels, 64)
        self.encoder2 = self.conv_block(64, 128)
        self.encoder3 = self.conv_block(128, 256)
        self.encoder4 = self.conv_block(256, 512)

        self.pool = nn.MaxPool2d(2)  # Downsamples by factor of 2

        # Bottleneck (middle part)
        self.mid = self.conv_block(512, 1024)

        # Decoder path ("expanding" path): upscales, combines with encoder outputs
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)  # Upsample
        self.dec4 = self.conv_block(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)

        # Final layer: reduces to 1 output channel per pixel (for binary segmentation)
        self.final = nn.Conv2d(64, 1, kernel_size=1)  # Output: logits per pixel

    def conv_block(self, in_ch, out_ch):
        """
        Helper function to build a block of two convolutional layers, 
        each followed by BatchNorm and ReLU activation.

        Args:
            in_ch (int): Number of input channels.
            out_ch (int): Number of output channels.
        """
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),  # Keeps spatial size the same
            nn.BatchNorm2d(out_ch),                   # Helps training
            nn.ReLU(inplace=True),                    # Non-linearity
            nn.Conv2d(out_ch, out_ch, 3, padding=1),  # Another conv layer
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        """
        Forward pass through the network.

        Args:
            x (torch.Tensor): Input tensor of shape [batch, channels, height, width]

        Returns:
            torch.Tensor: Output logits, shape [batch, 1, height, width]
        """
        # Encoder: save outputs for skip connections
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        m = self.mid(self.pool(e4))

        # Decoder: upsample, concatenate with encoder outputs (skip connections), then convolve
        d4 = self.dec4(torch.cat([self.up4(m), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.final(d1)  # Output shape: [batch, 1, H, W]


def dice_loss(pred, target, eps=1e-6):
    """
    Computes Dice Loss, which measures overlap between prediction and target.

    Args:
        pred (torch.Tensor): Raw logits from the model, shape [B, 1, H, W]
        target (torch.Tensor): Ground truth masks, shape [B, 1, H, W]
        eps (float): Small number to avoid division by zero.

    Returns:
        torch.Tensor: Dice loss (scalar)
    """
    # Apply sigmoid to logits to get probabilities between 0 and 1
    pred = torch.sigmoid(pred)
    # Intersection and union for each image in the batch
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    # Dice loss = 1 - Dice coefficient (higher is worse)
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()


def train_segmentation(model, train_loaders, val_loaders=None, epochs=10, lr=1e-3, device="cuda"):
    """
    Trains the segmentation model using both BCEWithLogitsLoss and Dice loss.

    Args:
        model (nn.Module): The segmentation model (U-Net).
        train_loaders (dict): Dictionary of DataLoader(s) for different patch sizes.
        val_loaders (dict or None): Not used in this code, can be used for validation.
        epochs (int): Number of training epochs.
        lr (float): Learning rate.
        device (str): 'cuda' for GPU or 'cpu' for CPU.

    During training, for each patch size, the function loops through the data in batches,
    computes the combined loss, and updates the model weights.
    """
    bce_loss = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss for logits
    model.to(device)                   # Move model to device (GPU/CPU)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)  # Stochastic Gradient Descent

    for epoch in range(epochs):
        model.train()  # Set model to training mode (affects layers like BatchNorm, Dropout)
        total_loss = 0.0
        print(f"Epoch {epoch + 1}/{epochs}")

        for size in train_loaders:
            loader = train_loaders[size]
            # Progress bar for this patch size
            pbar = tqdm(loader, desc=f"Training {size}x{size}", leave=False)

            for batch_idx, (x, y) in enumerate(pbar):
                x = x.to(device)
                y = y.to(device)

                optimizer.zero_grad()  # Clear previous gradients

                logits = model(x)  # [B, 1, H, W]
                logits_clamped = torch.clamp(logits, -20, 20)  # Clamp values for stability

                # Loss = BCE (pixelwise) + Dice (region overlap)
                loss = bce_loss(logits_clamped, y) + dice_loss(logits_clamped, y)

                if torch.isnan(loss):
                    print(f"❌ Batch {batch_idx}: loss is NaN, skipping")
                else:
                    loss.backward()      # Compute gradients
                    optimizer.step()     # Update weights
                    total_loss += loss.item()

        print(f"  ✅ Avg epoch loss: {total_loss:.4f}")


def build_dataloaders(X_dict, Y_dict, shuffle=True, regime='train'):
    """
    Builds PyTorch DataLoader objects for different patch sizes.

    Args:
        X_dict (dict): Dict of input tensors for each patch size.
        Y_dict (dict): Dict of label tensors for each patch size.
        shuffle (bool): Whether to shuffle the data (good for training).
        regime (str): Can be 'train' or 'test', not used in code.

    Returns:
        dict: Dictionary of DataLoader objects for each patch size.
    """
    batch_sizes = {128: 32, 256: 16}  # Set batch size for each patch size
    train_loaders = {}
    for size in X_dict:
        ds = PatchDataset(X_dict[size], Y_dict[size])  # Make dataset for this size
        train_loaders[size] = DataLoader(ds, batch_size=batch_sizes[size], shuffle=shuffle)
    return train_loaders


def evaluate_on_test(model, test_loaders, device="cuda", threshold=0.5):
    """
    Evaluates the segmentation model on test data and prints metrics.

    Args:
        model (nn.Module): The trained segmentation model.
        test_loaders (dict): Dictionary of DataLoader(s) for each patch size.
        device (str): 'cuda' for GPU or 'cpu' for CPU.
        threshold (float): Threshold for converting probabilities to binary masks.

    Prints:
        - IoU (Intersection over Union)
        - Dice coefficient
        - Precision
        - Recall
        - Image-level accuracy (if any pixel is detected as "rain" in image)
    Returns:
        float: Average of Dice coefficient and image-level accuracy.
    """
    model.eval()             # Set model to evaluation mode (no dropout/batchnorm update)
    model.to(device)

    total_iou = 0.0          # Intersection over Union accumulator
    total_dice = 0.0         # Dice coefficient accumulator
    total_prec = 0.0         # Precision accumulator
    total_recall = 0.0       # Recall accumulator
    total_batches = 0

    rain_y_true = []         # List for true image-level labels (rain/no rain)
    rain_y_pred = []         # List for predicted image-level labels

    with torch.no_grad():    # No need to compute gradients during evaluation
        for size, loader in test_loaders.items():
            pbar = tqdm(loader, desc=f"Test {size}x{size}", leave=False)
            for x, y in pbar:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)  # [B, 1, H, W]
                probs = torch.sigmoid(logits)        # Probabilities in [0, 1]
                preds = (probs > threshold).float()  # Binary predictions

                # Compute intersection and union for IoU
                intersection = (preds * y).sum(dim=(1, 2, 3))
                union = ((preds + y) > 0).float().sum(dim=(1, 2, 3))
                iou = (intersection / (union + 1e-6)).mean().item()

                # Compute Dice coefficient
                dice = (2 * intersection / (preds.sum(dim=(1,2,3)) + y.sum(dim=(1,2,3)) + 1e-6)).mean().item()

                # Precision and recall for all pixels
                tp = (preds * y).sum().item()                 # True positives
                fp = (preds * (1 - y)).sum().item()           # False positives
                fn = ((1 - preds) * y).sum().item()           # False negatives

                precision = tp / (tp + fp + 1e-6)
                recall = tp / (tp + fn + 1e-6)

                # Add to totals for averaging
                total_iou += iou
                total_dice += dice
                total_prec += precision
                total_recall += recall
                total_batches += 1

                # For image-level rain/no-rain: True if any pixel is "rain"
                rain_y_true += [(y > 0.5).any(dim=(1,2,3)).cpu()]
                rain_y_pred += [(preds > 0.5).any(dim=(1,2,3)).cpu()]

    if total_batches == 0:
        print("⚠️ No valid batches")
        return

    # Combine image-level labels for accuracy calculation
    rain_y_true = torch.cat(rain_y_true)
    rain_y_pred = torch.cat(rain_y_pred)
    acc = (rain_y_true == rain_y_pred).float().mean().item()

    dice_final = total_dice / total_batches

    print(f"\n📊 Test metrics across all sizes:")
    print(f"  • IoU   : {total_iou / total_batches:.4f}")
    print(f"  • Dice  : {dice_final:.4f}")
    print(f"  • Prec  : {total_prec / total_batches:.4f}")
    print(f"  • Recall: {total_recall / total_batches:.4f}")
    print(f"  • Image-level Rain Acc: {acc:.4f}")
    total_score = (dice_final + acc) / 2
    print(f"Final score: {total_score:.4f}")

    # Return a combined score for leaderboard
    return dice_final, acc, total_score

## Chargement et inspection des donnees


In [ ]:
# Charge depuis un fichier .npz
loaded = np.load(DATASET_PATH)

# Reconstruit les dictionnaires
X_train = {
    128: torch.from_numpy(loaded['X_train_128']),
    256: torch.from_numpy(loaded['X_train_256']),
}
Y_train = {
    128: torch.from_numpy(loaded['Y_train_128']),
    256: torch.from_numpy(loaded['Y_train_256']),
}
X_test = {
    128: torch.from_numpy(loaded['X_test_128']),
    256: torch.from_numpy(loaded['X_test_256']),
}
Y_test = {
    128: torch.from_numpy(loaded['Y_test_128']),
    256: torch.from_numpy(loaded['Y_test_256']),
}


del loaded

In [ ]:
for dx, dname in zip([X_train, Y_train],
                     ['X_train', 'Y_train']
                    ):
    for k, d in dx.items():
        print(dname, k, d.shape)

In [ ]:
sample_ind = 8

In [ ]:
plt.figure(figsize=(15,15))
for ind in range(16):
    plt.subplot(4,4,ind+1)
    plt.imshow(X_train[256][sample_ind, ind, ...])
    plt.colorbar()
    plt.contourf(Y_train[256][sample_ind, ...], cmap='Blues', alpha=.3)
    plt.axis('off')
    plt.title(ind)
plt.tight_layout()

In [ ]:
df = pd.read_csv(METADATA_PATH)

In [ ]:
df

In [ ]:
# Pour acceder aux metadonnees du sample_ind-ieme echantillon de X_train_128
i, j, start_time = df[(df['size'] == 128) & (df['split'] == 'train') & (df['ind'] == sample_ind)][['i', 'j', 'start_time']].values[0]

In [ ]:
print(df[(df['size'] == 128) & (df['split'] == 'train') & (df['ind'] == sample_ind)].values)

In [ ]:
solar_elevation(i, j, start_time)  # le soleil est sous l'horizon - visible sur les canaux 0 a 3

## Chargement du modele


In [ ]:
model = UNetSegmentation(in_channels=16)
model.load_state_dict(torch.load(BASELINE_MODEL, map_location=torch.device(DEVICE)))

In [ ]:
X_train[256][sample_ind:sample_ind+1, ...].to(torch.float32).shape

In [ ]:
prediction = model(X_train[256][sample_ind:sample_ind+1, ...].to(torch.float32))
prediction = torch.sigmoid(prediction[0,0,...].detach().cpu()).numpy()

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1,3,1)
plt.imshow(prediction > .5, cmap='Reds')
plt.title('Prediction')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(Y_train[256][sample_ind, ...], cmap='Blues')
plt.title('Truth')
plt.axis('off')

plt.subplot(1,3,3)
plt.title('Combined')
plt.imshow(prediction > .5, cmap='Reds', alpha=.5)
plt.contourf(Y_train[256][sample_ind, ...], cmap='Blues', alpha=.5)
plt.axis('off')

plt.tight_layout()

## Lancement de l'entrainement


In [ ]:
# Vous pouvez consulter model_utils.prepare_dataset pour la preparation des donnees
train_loaders = build_dataloaders(X_train, Y_train)
train_segmentation(model, train_loaders, None, lr=1e-3, epochs=1, device=DEVICE)  # changez de device si necessaire

## Evaluation du modele

Note : vous pouvez utiliser un modele separe pour la classification pluie/pas-pluie au niveau image.

Ici, une baseline est fournie : on s'appuie simplement sur les resultats de segmentation.



In [ ]:
test_loaders = build_dataloaders(X_test, Y_test)

In [ ]:
dice_val, accuracy, total = evaluate_on_test(model, test_loaders, device=DEVICE)

In [ ]:
total  # your task is to make it larger without peaking on Y_test

## Quels canaux peuvent aider a la detection de pluie ?

- C07, C13-C15 : temperature au sommet des nuages - des sommets froids signalent souvent des orages forts.
- C08-C10 : quantite de vapeur d'eau dans l'air.
- C04, C05, C06 : differenciation des types de nuage (glace vs eau).
- C11 : phase nuageuse et conditions poussiereuses.
- C16 : estimation de la hauteur des nuages (important pour les grands nuages de pluie).

| Canal | Type | Longueur d'onde | Utilite |
| --- | --- | --- | --- |
| **C01** | Visible | 0.47 um | Lumiere bleue : fumee, brume, petits nuages |
| **C02** | Visible | 0.64 um | Lumiere rouge : details des bords de nuage |
| **C03** | Proche IR | 0.86 um | Vegetation, phase nuageuse, contraste terre/eau |
| **C04** | Proche IR | 1.38 um | Nuages eleves fins (cirrus), haute atmosphere |
| **C05** | Proche IR | 1.61 um | Detection neige/nuage |
| **C06** | Proche IR | 2.25 um | Taille des particules nuageuses, contenu en glace |
| **C07** | IR | 3.90 um | Brouillard nocturne, chaleur de surface |
| **C08** | IR (VA) | 6.19 um | Vapeur d'eau haute |
| **C09** | IR (VA) | 6.95 um | Vapeur d'eau moyenne |
| **C10** | IR (VA) | 7.34 um | Vapeur d'eau basse |
| **C11** | IR | 8.50 um | Phase nuageuse, cendres volcaniques, poussiere |
| **C12** | IR | 9.61 um | Detection de l'ozone |
| **C13** | IR (propre) | 10.3 um | IR propre : sommets de nuages, air clair |
| **C14** | IR | 11.2 um | IR standard pour la temperature au sommet |
| **C15** | IR | 12.3 um | Fenetre "sale" : nuages profonds, vapeur d'eau |
| **C16** | IR (CO2) | 13.3 um | Bande CO2 : estimation de la hauteur des nuages |



## Soumission

Votre notebook doit generer un `submission.zip` contenant vos predictions pour le jeu de test public `pred_a.npz` et le jeu prive `pred_b.npz`. Chaque `.npz` doit contenir `Y_pred_128` (forme $51\times128\times128$) et `Y_pred_256` (forme $183\times256\times256$).


In [ ]:
# genere d'abord pred_a.npz

model.eval()
model.to(DEVICE)

Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i]
        metadata = df[(df['size'] == 128) & (df['split'] == 'test') & (df['ind'] == i)] # exemple d'utilisation des metadonnees
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i]
        metadata = df[(df['size'] == 256) & (df['split'] == 'test') & (df['ind'] == i)]
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# Vos tableaux de prediction DOIVENT s'appeler `Y_pred_128` et `Y_pred_256`, et le fichier `pred_a.npz` pour le classement public
np.savez('pred_a.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)

In [ ]:
# Votre notebook accedera au jeu de test via la variable d'environnement DATA_PATH apres soumission.
# Pendant le debogage, il est normal que ce bloc echoue. 
# DATA_PATH est une variable d'environnement fournie par la machine de test, utilisee pour lire le jeu de test. 
# Les participants ne peuvent pas y acceder directement. Conservez cette variable pendant la soumission, sinon la machine de test ne pourra pas lire le jeu de test.
TEST_PATH = os.environ.get('DATA_PATH', "test")

test = np.load(Path(TEST_PATH) / "X_test.npz") # Lit les donnees de test depuis X_test.npz dans le chemin fourni

X_test = {
    128: torch.from_numpy(test['X_test_128']),
    256: torch.from_numpy(test['X_test_256']),
} # seul X_test est fourni

df_test = pd.read_csv(Path(TEST_PATH) / "metadata_test.csv") # Lit les metadonnees depuis metadata_test.csv dans le chemin fourni

In [ ]:
Y_pred_128 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[128]))):
        x = X_test[128][i]
        metadata = df_test[(df_test['size'] == 128) & (df_test['ind'] == i)] # exemple d'utilisation des metadonnees
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_128.append(preds.cpu().detach().numpy())
Y_pred_128 = np.concatenate(Y_pred_128, axis=0)

print(Y_pred_128.shape)

Y_pred_256 = []
with torch.no_grad():
    for i in tqdm(range(len(X_test[256]))):
        x = X_test[256][i]
        metadata = df_test[(df_test['size'] == 256) & (df_test['ind'] == i)]
        logits = model(x.unsqueeze(0).to(torch.float32).to(DEVICE))
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float().squeeze(0)
        Y_pred_256.append(preds.cpu().detach().numpy())
Y_pred_256 = np.concatenate(Y_pred_256, axis=0)

print(Y_pred_256.shape)

# Nommez `Y_pred_128` et `Y_pred_256`, et le fichier `pred_b.npz` pour le classement prive
np.savez('pred_b.npz', Y_pred_128=Y_pred_128, Y_pred_256=Y_pred_256)


In [ ]:
# compresse `pred_a.npz` et `pred_b.npz` dans `submission.zip`
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('pred_a.npz')
    zipf.write('pred_b.npz')
